In [1]:
import numpy as np
import pandas as pd
pd.options.mode.chained_assignment = None
import time
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, make_scorer,classification_report,accuracy_score
from sklearn import linear_model, tree, ensemble
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
#from ensampling.bagging import Blagging
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

/usr/local/lib/python3.7/dist-packages/sklearn/externals/six.py:31: FutureWarning: The module is deprecated in version 0.21 and will be removed in version 0.23 since we've dropped support for Python 2.7. Please rely on the official version of six (https://pypi.org/project/six/).
  "(https://pypi.org/project/six/).", FutureWarning)
/usr/local/lib/python3.7/dist-packages/sklearn/utils/deprecation.py:144: FutureWarning: The sklearn.neighbors.base module is  deprecated in version 0.22 and will be removed in version 0.24. The corresponding classes / functions should instead be imported from sklearn.neighbors. Anything that cannot be imported from sklearn.neighbors is now part of the private API.
  warnings.warn(message, FutureWarning)


In [3]:
from google.colab import drive
drive.mount('/content/drive')
import seaborn as sb
import pandas as pd
import numpy as np
import math 

Mounted at /content/drive


In [7]:
n_runs=30 # Define the number of models to be trained
scorer = make_scorer(average_precision_score, needs_threshold=True, average="micro",)#make_scorer(cohen_kappa_score)#'roc_auc' 

min_samples_leaf=5
n_estimators=10
criterion='entropy'
max_depth=[15]
dataset='kaggle' #'data/bopredict.csv'
n_folds=5
save_run=10

#df = pd.read_csv('data/'+dataset+'.csv')
df = pd.read_csv('/content/drive/MyDrive/kaggle.csv')
X = df.drop(['went_on_backorder','sku'],axis=1).values
y = df['went_on_backorder'].values

print("dataset:",dataset)

dataset: kaggle


In [8]:
estimators = [
("Adaptive Boosting-u", "adaboost-u",
     Pipeline([
         ('res', RandomUnderSampler()),
         ('tree', ensemble.AdaBoostClassifier(n_estimators=10)),
               ]),
     {
      }),
("Adaptive Boosting", 'adaboost',
     ensemble.AdaBoostClassifier(n_estimators=10),
    {
    }),


("GradientBoosting-u", "gb-u",
     Pipeline([
         ('res', RandomUnderSampler()),
         ('tree', ensemble.GradientBoostingClassifier(n_estimators=n_estimators,
                                         min_samples_leaf=min_samples_leaf)),
               ]),
     {
      }),
     ("Decision Tree-u", 'dt-u',
      Pipeline([('res', RandomUnderSampler()),
                ('tree', tree.DecisionTreeClassifier(
                    min_samples_leaf=min_samples_leaf, criterion=criterion))
                ]),
      {'tree__max_depth': max_depth,
       }),

    ("Decision Tree", 'cart',
     tree.DecisionTreeClassifier(min_samples_leaf=min_samples_leaf,
                                 criterion=criterion),
    {'max_depth':max_depth,
     #'max_features':[3,5,10,None],
     #'splitter':['best','random'],
     'criterion':['entropy','gini'],
    }),
    ("RandomForest", "rf",
     ensemble.RandomForestClassifier(n_estimators=n_estimators,
            min_samples_leaf=min_samples_leaf, criterion=criterion),
    {'max_depth':max_depth,
    }),
    ("RandomForest-u", "rf-u",
     Pipeline([('res', RandomUnderSampler()),
               ('tree',ensemble.RandomForestClassifier(n_estimators=n_estimators,
            min_samples_leaf=min_samples_leaf, criterion=criterion)),
              ]),
    {
    }),

    ("GradientBoosting", "gb",
     ensemble.GradientBoostingClassifier(n_estimators=n_estimators,
            min_samples_leaf=min_samples_leaf),
    {'max_depth':[10],
    })
]

In [9]:
for est_full_name, est_name, est, params in estimators:
    print ('\n%s\n%s\n' % ('-'*25, est_full_name))
    print ('Run\tEst\tScore\tAUROC\tAUPRC\tTime\tBest parameters')
    matriz = []
    t0 = time.time()
    X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y,test_size=0.15, random_state=32)
    kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=int(8* 9))
    gs = GridSearchCV(est, params, cv=kf,  # n_iter=n_iter_search,
                      scoring=scorer, verbose=0, n_jobs=-1)
    t1 = time.time()
    gs.fit(X_train, y_train)

    sm = SMOTE(random_state=2)
    X_test, y_test= sm.fit_sample(X_test, y_test)

    y_prob0 = gs.best_estimator_.predict_proba(X_train)[:, 1]
    y_prob = gs.best_estimator_.predict_proba(X_test)[:, 1]
    y_pred = gs.best_estimator_.predict(X_test)

    roc = roc_auc_score(y_test, y_prob)
    pr = average_precision_score(y_test, y_prob)
    acc = accuracy_score(y_test,y_pred)
    print(acc)

    classif_report = classification_report(y_test, y_pred)
    print(classif_report)

    run_time = time.time() - t1
    avg_time = run_time / gs.n_splits_

    print("\t%s\t%.4f\t%.4f\t%.4f\t%.2f\t%s" % (est_name,
                                                  gs.best_score_, roc, pr, avg_time, gs.best_params_))

    matriz.append(
        {
         'estimator': est_name,
         'roc': roc,
         'pr': pr,
         'best_params': gs.best_params_,
         'avg_time': avg_time,
         })

 
    print("Elapsed time: %0.3fs" % (time.time()-t0))
    # Save results
    data = pd.DataFrame(matriz)


-------------------------
Adaptive Boosting-u

Run	Est	Score	AUROC	AUPRC	Time	Best parameters


/usr/local/lib/python3.7/dist-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function safe_indexing is deprecated; safe_indexing is deprecated in version 0.22 and will be removed in version 0.24.
  warnings.warn(msg, category=FutureWarning)
/usr/local/lib/python3.7/dist-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function safe_indexing is deprecated; safe_indexing is deprecated in version 0.22 and will be removed in version 0.24.
  warnings.warn(msg, category=FutureWarning)


0.8591045742082298
              precision    recall  f1-score   support

           0       0.87      0.85      0.86    287394
           1       0.85      0.87      0.86    287394

    accuracy                           0.86    574788
   macro avg       0.86      0.86      0.86    574788
weighted avg       0.86      0.86      0.86    574788

	adaboost-u	0.1020	0.9287	0.9145	4.16	{}
Elapsed time: 22.360s

-------------------------
Adaptive Boosting

Run	Est	Score	AUROC	AUPRC	Time	Best parameters


/usr/local/lib/python3.7/dist-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function safe_indexing is deprecated; safe_indexing is deprecated in version 0.22 and will be removed in version 0.24.
  warnings.warn(msg, category=FutureWarning)


0.5076428178737205
              precision    recall  f1-score   support

           0       0.50      1.00      0.67    287394
           1       0.96      0.02      0.03    287394

    accuracy                           0.51    574788
   macro avg       0.73      0.51      0.35    574788
weighted avg       0.73      0.51      0.35    574788

	adaboost	0.1097	0.9255	0.9106	33.07	{}
Elapsed time: 166.932s

-------------------------
GradientBoosting-u

Run	Est	Score	AUROC	AUPRC	Time	Best parameters


...

0.5045982170817762
              precision    recall  f1-score   support

           0       0.50      1.00      0.67    287394
           1       1.00      0.01      0.02    287394

    accuracy                           0.50    574788
   macro avg       0.75      0.50      0.34    574788
weighted avg       0.75      0.50      0.34    574788

	rf	0.2440	0.9437	0.9451	48.73	{'max_depth': 15}
Elapsed time: 245.159s

-------------------------
RandomForest-u

Run	Est	Score	AUROC	AUPRC	Time	Best parameters


/usr/local/lib/python3.7/dist-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function safe_indexing is deprecated; safe_indexing is deprecated in version 0.22 and will be removed in version 0.24.
  warnings.warn(msg, category=FutureWarning)
/usr/local/lib/python3.7/dist-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function safe_indexing is deprecated; safe_indexing is deprecated in version 0.22 and will be removed in version 0.24.
  warnings.warn(msg, category=FutureWarning)


0.882455792396501
              precision    recall  f1-score   support

           0       0.89      0.87      0.88    287394
           1       0.87      0.89      0.88    287394

    accuracy                           0.88    574788
   macro avg       0.88      0.88      0.88    574788
weighted avg       0.88      0.88      0.88    574788

	rf-u	0.1447	0.9458	0.9383	3.37	{}
Elapsed time: 18.577s

-------------------------
GradientBoosting

Run	Est	Score	AUROC	AUPRC	Time	Best parameters


/usr/local/lib/python3.7/dist-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function safe_indexing is deprecated; safe_indexing is deprecated in version 0.22 and will be removed in version 0.24.
  warnings.warn(msg, category=FutureWarning)


0.5303050863970716
              precision    recall  f1-score   support

           0       0.52      1.00      0.68    287394
           1       0.98      0.06      0.12    287394

    accuracy                           0.53    574788
   macro avg       0.75      0.53      0.40    574788
weighted avg       0.75      0.53      0.40    574788

	gb	0.1774	0.9413	0.9373	222.95	{'max_depth': 10}
Elapsed time: 1116.252s
